# Lab 2.1 &mdash; Chain-of-Thought, Measured

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Score a direct answer and a reasoned answer on the same cases
- Separate the answer from the reasoning that led to it
- Put a number on what the extra tokens bought

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **The thread continues.** Same synthetic payment-exception case file as Module 1.
> Module 1 asked whether to build an agent; Module 2 asks how it should think.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

Chain-of-Thought is a prompt, not an architecture change: ask the model to work in steps before it
answers. It usually helps on multi-step questions, it always costs tokens, and the reasoning it
shows you is **a rationalisation, not a transcript** &mdash; useful evidence for a human reviewer,
never a control.

This lab does the only thing that settles it: run both on the same cases and compare.

## Section 1 &mdash; Separate the answer from the reasoning

If you cannot extract the answer reliably, you cannot score it. Reasoning ends and the answer
begins at a marker you impose &mdash; here, a final line starting `ANSWER:`.

In [ ]:
ANSWER_MARKER = "ANSWER:"

def split_reasoning(text: str) -> tuple[str, str]:
    """Return (reasoning, answer) from a model reply.

    The answer is the text after the LAST occurrence of ANSWER_MARKER, stripped.
    If the marker never appears, the reasoning is empty and the whole reply is the answer --
    degrade, never raise.
    """
    if ANSWER_MARKER not in text:
        return "", text.strip()
    head, _, tail = text.rpartition(ANSWER_MARKER)
    return head.strip(), tail.strip()

In [ ]:
# --- Self-check: Section 1
_reply = "Step 1: the code is LIMIT_BREACH.\nStep 2: policy needs Treasury.\nANSWER: Treasury approval"

check("the answer is taken from after the marker",
      lambda: split_reasoning(_reply)[1] == "Treasury approval")
check("the reasoning is everything before it",
      lambda: "Step 1" in split_reasoning(_reply)[0])
check("a reply with no marker still yields an answer",
      lambda: split_reasoning("Treasury approval")[1] == "Treasury approval",
      "return ('', text.strip()) rather than raising")
check("a reply with no marker has empty reasoning",
      lambda: split_reasoning("Treasury approval")[0] == "")
check("the LAST marker wins",
      lambda: split_reasoning("ANSWER: draft\nrethinking\nANSWER: final")[1] == "final",
      "use rpartition, not partition -- models restate the marker")

## Section 2 &mdash; The two prompts

Identical task, identical cases. The only difference is whether the model is asked to work in
steps. Keep everything else fixed or the comparison means nothing.

In [ ]:
TASK = ("You are a payments operations analyst. Given a payment record and the policy catalogue, "
        "say who must action the exception: OPERATIONS, TREASURY, COMPLIANCE or ORIGINATOR.")

DIRECT_PROMPT = TASK + "\nReply with a single line: ANSWER: <one of the four>"

def build_cot_prompt() -> str:
    """The chain-of-thought prompt: same task, same answer format, plus stepwise reasoning."""
    instruction = (
        "Work through it in numbered steps first: state the reason code, then the policy that "
        "applies, then who that policy makes responsible. "
        "Finish with a single final line: ANSWER: <one of the four>"
    )
    return TASK + "\n" + instruction

In [ ]:
# --- Self-check: Section 2
check("both prompts demand the same answer format",
      lambda: ANSWER_MARKER in DIRECT_PROMPT and ANSWER_MARKER in build_cot_prompt(),
      "if the formats differ you are measuring your parser, not the reasoning")
check("only the chain-of-thought prompt asks for steps",
      lambda: "step" in build_cot_prompt().lower() and "step" not in DIRECT_PROMPT.lower())
check("both carry the identical task definition",
      lambda: DIRECT_PROMPT.startswith(TASK) and build_cot_prompt().startswith(TASK),
      "change one variable at a time or the result is uninterpretable")

## Section 3 &mdash; The eval set and the scorer

Six cases, each with the responsible party known in advance. Two are deliberately awkward: one has
no exception at all, and one references a payment that does not exist.

In [ ]:
CASES = [
    {"ref": "PMT-1002", "expect": "OPERATIONS"},   # insufficient funds -> retry, ops desk
    {"ref": "PMT-1003", "expect": "TREASURY"},     # limit breach -> treasury approval
    {"ref": "PMT-1004", "expect": "ORIGINATOR"},   # invalid IBAN -> return to originator
    {"ref": "PMT-1005", "expect": "COMPLIANCE"},   # sanctions review -> compliance decides
    {"ref": "PMT-1001", "expect": "OPERATIONS"},   # settled: no exception to action
    {"ref": "PMT-9999", "expect": "OPERATIONS"},   # unknown reference: cannot be actioned blind
]

def render_case(ref: str) -> str:
    """The case text handed to the model -- identical for both prompts."""
    rec = LEDGER.get(ref)
    if rec is None:
        return f"PAYMENT {ref}: not found in the ledger."
    policy = POLICY.get(rec["reason_code"], "no policy on file")
    return f"PAYMENT {ref}: {json.dumps(rec)}\nPOLICY: {policy}"

def scores(answer: str, expect: str) -> bool:
    """A case passes when the expected party is named in the answer, case-insensitively."""
    return expect.lower() in answer.lower()

In [ ]:
# --- Self-check: Section 3
check("an exact answer passes", lambda: scores("ANSWER: TREASURY", "TREASURY") is True)
check("case does not matter", lambda: scores("answer: treasury", "TREASURY") is True)
check("a wrong party fails", lambda: scores("COMPLIANCE", "TREASURY") is False)
check("every case renders without raising",
      lambda: all(isinstance(render_case(c["ref"]), str) for c in CASES))
check("the unknown reference renders as not found",
      lambda: "not found" in render_case("PMT-9999"))

## Run it for real

Both prompts, all six cases, one table. This is the first real measurement of the module.

In [ ]:
def run_arm(prompt: str, label: str) -> dict:
    """Run every case under one prompt. Returns pass rate and rough token cost."""
    hits, chars = 0, 0
    for c in CASES:
        reply = ask(render_case(c["ref"]), system=prompt)
        _, answer = split_reasoning(reply)
        ok = scores(answer, c["expect"])
        hits += 1 if ok else 0
        chars += len(prompt) + len(render_case(c["ref"])) + len(reply)
        print(f"  {c['ref']}  expect {c['expect']:11} got {answer[:34]:34} {'PASS' if ok else 'FAIL'}")
    return {"arm": label, "pass_rate": round(hits / len(CASES), 3), "est_tokens": chars // 4}

if llm_ready():
    try:
        print("--- direct ---");            direct = run_arm(DIRECT_PROMPT, "direct")
        print("\n--- chain-of-thought ---"); cot = run_arm(build_cot_prompt(), "cot")
        print(f"\n{'arm':22}{'pass rate':>12}{'est tokens':>13}")
        for r in (direct, cot):
            print(f"{r['arm']:22}{r['pass_rate']:>12}{r['est_tokens']:>13}")
        gain = cot["pass_rate"] - direct["pass_rate"]
        mult = cot["est_tokens"] / max(direct["est_tokens"], 1)
        print(f"\nchain-of-thought bought {gain:+.2f} pass rate for {mult:.1f}x the tokens.")
    except NameError:
        print("(fill in the blanks above, then re-run this cell)")

### Read it

Three things to look at, in this order:

1. **Did the gain justify the multiple?** On a task this small it often does not &mdash; and that is a
   real result, not a failed lab.
2. **Where did the direct arm fail?** Usually the two awkward cases, which is exactly where working
   in steps helps.
3. **Read one reasoning trace against its answer.** If a case passed with reasoning that does not
   support it, you have just seen why the reasoning text is evidence and not a control.

In [ ]:
score()

## Your turn

1. Add a case the direct arm gets right and the reasoned arm gets wrong. They exist &mdash; extra steps
   give a model more places to talk itself out of a correct first instinct.
2. `scores()` does substring matching, so an answer of "not TREASURY" would pass. Tighten it, then
   decide whether the stricter scorer changes the verdict. If a scoring change flips your
   conclusion, the conclusion was never solid.